<a href="https://colab.research.google.com/github/youngPath12/AI-Health-Bio-Data-Team-4-Professor-yoon-/blob/Hyein-Yu/brain_age_gap_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MRI 기반 Brain Age Gap 예측을 통한 알츠하이머 취약성 분석

## 연구 질문
정상 인지군의 MRI 구조 지표로 실제 나이를 예측하는 뇌 연령 모델을 만듭니다. 이후 MCI와 AD 환자에게 적용해 예측 뇌 나이와 실제 나이의 차이인 Brain Age Gap(BAG)을 비교합니다.

BAG = 예측 뇌 나이 - 실제 나이

- 양수 BAG: 실제 나이보다 뇌가 더 늙어 보이는 경향
- 음수 BAG: 실제 나이보다 뇌가 더 젊어 보이는 경향

> 교육·포트폴리오용 연구입니다. 개인 진단이나 치료 결정에 사용하면 안 됩니다.

사용 데이터: ADSP_PHC.zip 내부의 ADSP_PHC_T1_FS_22Jan2026.csv. 이 파일에는 MRI로부터 추출한 해마·뇌실·피질 구조 수치, MRI 시점 나이, 진단 그룹이 있습니다.

## 0. Colab 준비

아래 셀을 실행한 뒤 뜨는 파일 선택 창에서 프로젝트 폴더의 ADSP_PHC.zip을 선택하세요. 압축을 직접 풀 필요는 없습니다.

> GitHub에는 이 노트북과 README만 올립니다. ADNI 원본 ZIP/CSV는 재배포하지 마세요.

In [ ]:
!pip -q install seaborn
!apt-get -qq update && apt-get -qq install fonts-nanum

import zipfile
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from scipy.stats import kruskal

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('준비 완료')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-nanum.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...


## 1. 데이터 업로드

PHC_Age_T1은 MRI 촬영 시점의 나이이고, PHC_Diagnosis는 진단 그룹입니다. 보통 이 데이터의 진단 코드는 1=정상군, 2=MCI, 3=AD입니다. 업로드 후 실제 분포를 확인합니다.

In [ ]:
uploaded = files.upload()  # ADSP_PHC.zip을 선택하세요
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('ADSP_PHC.zip 파일을 업로드해 주세요.')

extract_dir = Path('/content/adsp_phc')
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

candidates = list(extract_dir.rglob('ADSP_PHC_T1_FS_*.csv'))
if not candidates:
    raise FileNotFoundError('ADSP_PHC_T1_FS CSV 파일을 찾지 못했습니다.')
data_path = candidates[0]
df = pd.read_csv(data_path, low_memory=False)
print(f'불러온 파일: {data_path.name}')
print(f'행 수: {len(df):,}, 열 수: {df.shape[1]:,}')
display(df[['RID', 'PHC_Age_T1', 'PHC_Diagnosis', 'PHC_Sex']].head())
display(df['PHC_Diagnosis'].value_counts(dropna=False).sort_index().rename('참가자 수'))

## 2. MRI 변수 선택

MRI 파일에는 수백 개의 수치가 있습니다. 이번 프로젝트에서는 해마, 뇌실, 피질, 내후각피질, 방추상회, 측두엽 등 뇌 노화와 관련성이 잘 알려진 구조 지표를 자동 선택합니다. 결측치는 나중에 각 변수의 중앙값으로 채웁니다.

In [ ]:
data = df.copy()
data['PHC_Age_T1'] = pd.to_numeric(data['PHC_Age_T1'], errors='coerce')
data['PHC_Diagnosis'] = pd.to_numeric(data['PHC_Diagnosis'], errors='coerce')
data = data.dropna(subset=['PHC_Age_T1', 'PHC_Diagnosis']).copy()

brain_keywords = ['Hippocampus', 'Ventricle', 'Cortex', 'BrainSeg', 'entorhinal', 'fusiform', 'middletemporal', 'precuneus', 'amygdala', 'temporal']
exclude_cols = {'RID', 'PTID', 'SUBJID', 'PHC_LONIUID', 'PHC_Age_T1', 'PHC_Age_Cognition', 'PHC_Diagnosis', 'PHC_Visit'}
numeric_cols = data.select_dtypes(include=np.number).columns
feature_candidates = [c for c in numeric_cols if c not in exclude_cols and any(word.lower() in c.lower() for word in brain_keywords)]
feature_cols = [c for c in feature_candidates if data[c].isna().mean() < 0.40][:80]
if len(feature_cols) < 10:
    raise ValueError('MRI 특징이 너무 적습니다. feature_cols를 확인해 주세요.')
print(f'사용 MRI 변수 수: {len(feature_cols)}')
print(feature_cols[:20])
display((data[feature_cols].isna().mean() * 100).sort_values(ascending=False).head(15).to_frame('결측 비율(%)').round(1))

## 3. 정상군과 환자군 정의

설계서의 핵심은 정상군만으로 나이 예측 모델을 학습하는 것입니다. 같은 사람이 여러 MRI를 갖는 경우 가장 이른 MRI 한 건만 사용합니다. 앞 단계에서 확인한 진단 코드가 다르면 diagnosis_map을 수정하세요.

In [ ]:
diagnosis_map = {1: 'CN (정상)', 2: 'MCI', 3: 'AD'}
data['group'] = data['PHC_Diagnosis'].map(diagnosis_map)
analysis_df = data[data['group'].isin(diagnosis_map.values())].copy()
if 'PHC_SCANDATE' in analysis_df.columns:
    analysis_df['PHC_SCANDATE'] = pd.to_datetime(analysis_df['PHC_SCANDATE'], errors='coerce')
    analysis_df = analysis_df.sort_values('PHC_SCANDATE')
analysis_df = analysis_df.drop_duplicates(subset='RID', keep='first').copy()

display(analysis_df.groupby('group')['PHC_Age_T1'].agg(['count', 'mean', 'std']).round(1))
sns.countplot(data=analysis_df, x='group', order=['CN (정상)', 'MCI', 'AD'], hue='group', legend=False)
plt.title('분석 대상 진단 그룹 분포')
plt.xlabel('진단 그룹')
plt.ylabel('참가자 수')
plt.show()

## 4. 정상군으로 뇌 나이 모델 학습

정상군의 80%로 학습하고 남은 20%로 평가합니다. ExtraTreesRegressor는 여러 결정트리를 합친 모델로, 표 형태 MRI 수치에서 복잡한 관계를 학습합니다.

- MAE: 예측 나이가 실제 나이와 평균 몇 살 차이 나는지
- R²: 실제 나이 차이를 모델이 얼마나 설명하는지 (1에 가까울수록 좋음)

In [ ]:
controls = analysis_df[analysis_df['group'] == 'CN (정상)'].copy()
patients = analysis_df[analysis_df['group'].isin(['MCI', 'AD'])].copy()
X_control = controls[feature_cols]
y_control = controls['PHC_Age_T1']
X_train, X_holdout, y_train, y_holdout = train_test_split(X_control, y_control, test_size=0.20, random_state=RANDOM_STATE)

brain_age_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', ExtraTreesRegressor(n_estimators=400, min_samples_leaf=2, max_features=0.8, random_state=RANDOM_STATE, n_jobs=-1))
])
brain_age_model.fit(X_train, y_train)
holdout_pred = brain_age_model.predict(X_holdout)
mae = mean_absolute_error(y_holdout, holdout_pred)
rmse = mean_squared_error(y_holdout, holdout_pred, squared=False)
r2 = r2_score(y_holdout, holdout_pred)
print(f'정상군 검증 MAE: {mae:.2f}세')
print(f'정상군 검증 RMSE: {rmse:.2f}세')
print(f'정상군 검증 R²: {r2:.3f}')

plt.figure(figsize=(6, 6))
plt.scatter(y_holdout, holdout_pred, alpha=0.7)
limits = [min(y_holdout.min(), holdout_pred.min()), max(y_holdout.max(), holdout_pred.max())]
plt.plot(limits, limits, 'r--', label='완벽한 예측')
plt.xlabel('실제 나이')
plt.ylabel('예측 뇌 나이')
plt.title('정상군: 실제 나이와 예측 뇌 나이')
plt.legend()
plt.show()

## 5. Brain Age Gap 계산과 그룹 비교

모델 학습에 쓰지 않은 정상군(holdout)과 MCI·AD 환자군에만 모델을 적용합니다. Kruskal-Wallis 검정의 p값이 0.05보다 작으면 세 그룹의 BAG 분포가 모두 같다고 보기 어렵다는 뜻입니다. 이 결과는 연관성이지 인과관계는 아닙니다.

In [ ]:
holdout_controls = controls.loc[X_holdout.index].copy()
comparison_df = pd.concat([holdout_controls, patients], axis=0).copy()
comparison_df['predicted_brain_age'] = brain_age_model.predict(comparison_df[feature_cols])
comparison_df['brain_age_gap'] = comparison_df['predicted_brain_age'] - comparison_df['PHC_Age_T1']

group_order = ['CN (정상)', 'MCI', 'AD']
bag_summary = comparison_df.groupby('group')['brain_age_gap'].agg(['count', 'mean', 'median', 'std']).reindex(group_order)
display(bag_summary.round(2))
samples = [comparison_df.loc[comparison_df['group'] == g, 'brain_age_gap'].dropna() for g in group_order]
stat, p_value = kruskal(*samples)
print(f'Kruskal-Wallis H={stat:.3f}, p={p_value:.4g}')

plt.figure(figsize=(9, 5))
sns.boxplot(data=comparison_df, x='group', y='brain_age_gap', order=group_order, hue='group', legend=False, palette='Set2')
sns.stripplot(data=comparison_df, x='group', y='brain_age_gap', order=group_order, color='black', alpha=0.20, size=3)
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='BAG = 0')
plt.title('진단 그룹별 Brain Age Gap')
plt.xlabel('진단 그룹')
plt.ylabel('Brain Age Gap (예측 뇌 나이 - 실제 나이, 세)')
plt.legend()
plt.show()

## 6. 나이 예측에 중요한 뇌 영역

Permutation Importance는 특정 MRI 변수를 섞었을 때 모델 성능이 얼마나 떨어지는지 계산합니다. 많이 떨어질수록 모델의 나이 예측에 중요한 변수입니다. 중요한 변수는 모델 예측에 유용한 연관 변수이지 노화를 직접 일으킨 원인이라는 뜻은 아닙니다.

In [ ]:
importance = permutation_importance(brain_age_model, X_holdout, y_holdout, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring='neg_mean_absolute_error')
importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importance.importances_mean}).sort_values('importance', ascending=False).head(15)
display(importance_df)
plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x='importance', y='feature', hue='feature', legend=False, palette='viridis')
plt.title('뇌 나이 예측에 중요한 MRI 구조 지표 Top 15')
plt.xlabel('변수를 섞었을 때의 성능 감소량')
plt.ylabel('MRI 변수')
plt.show()

## 7. 포트폴리오·발표 정리

발표 흐름: 정상군 MRI로 뇌 나이 모델 구축 → 정상군 MAE/R² 제시 → MCI·AD의 BAG 분포 비교 → 중요한 MRI 구조 변수 해석 → 한계 설명.

한계: ADNI 표본의 대표성 제한, 외부 검증 부재, 단면 자료 중심, BAG의 연령 편향 가능성.

| 담당 | 업무 |
|---|---|
| 1 | ADNI 파일·변수 사전 확인과 데이터 불러오기 |
| 2 | 결측치·연령·진단 그룹 EDA와 시각화 |
| 3 | Brain Age 모델 학습과 MAE/R² 평가 |
| 4 | BAG 그룹 비교, 중요 변수 해석, README·발표 구성 |

GitHub에는 노트북/README만 올리고 ADSP_PHC.zip과 추출 CSV는 절대 올리지 마세요.